# 网络流量时间序列预测系统

本 notebook 将项目原型中的数据预处理、窗口构建、模型训练、模型评估、对比实验、消融实验和可视化统一到一个文件中。

核心约束：
- 时间序列严格按 `date` 从小到大切分 train / val / test。
- `log1p` 变换后的标准化参数只在训练集上拟合，再应用到验证集和测试集。
- 滑动窗口只在各自 split 内生成，不允许窗口跨越 train / val / test 边界。
- 默认建模输入只使用历史 `n_flows`，不使用未来网络统计字段，避免引入业务预测时不可获得的信息。
- 最终模型只根据验证集 MSE 选择，测试集只用于最终评估。


## 0. 环境配置


In [ ]:
# 安装依赖。
!pip install numpy pandas torch tqdm matplotlib seaborn scikit-learn


## 1. 基础配置


In [2]:
from pathlib import Path
from dataclasses import dataclass
from collections import defaultdict
from IPython.display import display
import json
import math
import os
import random
import shutil
import sys
import time
import warnings

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
CUDA_VISIBLE_DEVICES = "0"
# 必须在 import torch 前设置，方便后续指定训练显卡。
os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_VISIBLE_DEVICES
warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

try:
    import seaborn as sns
except Exception:
    sns = None

plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "WenQuanYi Micro Hei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 140


def locate_project_root():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "AGENTS.md").exists() and (base / "raw_data.csv").exists():
            return base
    return Path.cwd()


PROJECT_ROOT = locate_project_root()


class CONFIG:
    is_debug = False
    seed = 308
    cuda_visible_devices = CUDA_VISIBLE_DEVICES

    raw_csv = PROJECT_ROOT / "raw_data.csv"
    time_col = "date"
    target_col = "n_flows"
    freq = "10min"
    freq_minutes = 10

    input_len = 168
    pred_len = 72
    stride = 1
    train_ratio = 0.70
    val_ratio = 0.15
    use_target_only = True

    data_root = PROJECT_ROOT / "data" / "network_traffic_forecast"
    processed_dir = data_root / "processed"
    window_dir = data_root / f"window_samples_in{input_len}_out{pred_len}"

    output_root = PROJECT_ROOT / "outputs" / "network_traffic_forecast"
    run_dir = output_root / "runs"
    figure_dir = output_root / "figures"
    model_dir = output_root / "models"
    metric_dir = output_root / "metrics"
    prediction_dir = output_root / "predictions"
    log_dir = output_root / "logs"

    device = "cuda" if torch.cuda.is_available() else "cpu"
    n_workers = 0
    batch_size = 128
    epochs = 40
    n_runs = 1
    lr = 1e-4
    weight_decay = 1e-4
    grad_clip = 1.0
    patience = 8
    min_delta = 1e-5
    lr_patience = 3
    lr_factor = 0.5
    min_lr = 1e-6

    d_model = 128
    n_heads = 8
    e_layers = 2
    d_ff = 256
    dropout = 0.1
    patch_len = 16
    patch_stride = 8
    ma_kernel = 25
    fed_modes = 32
    use_revin = False

    plot_samples = 6
    max_scatter_points = 60000
    force_rebuild_data = False
    run_full_experiment = True

    comparison_models = [
        "DLinear",
        "Transformer",
        "Autoformer",
        "TimeXer",
        "FEDformer",
        "iTransformer",
        "PatchTST",
        "SDF-PatchTST",
    ]
    ablation_models = [
        "PatchTST",
        "SDF-PatchTST",
        "SDF-PatchTST w/o Decomp",
        "SDF-PatchTST w/o Spectral",
    ]
    run_models = list(dict.fromkeys(comparison_models + ablation_models))
    main_model_name = "SDF-PatchTST"

    # 快速联调开关：只跑少量样本和少量 epoch，便于检查环境。
    smoke_max_samples = 512


if CONFIG.is_debug:
    CONFIG.epochs = 2
    CONFIG.n_runs = 1
    CONFIG.batch_size = 128
    CONFIG.run_models = ["DLinear", "PatchTST", "SDF-PatchTST"]
    CONFIG.comparison_models = CONFIG.run_models
    CONFIG.ablation_models = ["PatchTST", "SDF-PatchTST"]


def mkdirs():
    for p in [
        CONFIG.data_root,
        CONFIG.processed_dir,
        CONFIG.window_dir,
        CONFIG.output_root,
        CONFIG.run_dir,
        CONFIG.figure_dir,
        CONFIG.figure_dir / "data",
        CONFIG.figure_dir / "training",
        CONFIG.figure_dir / "results",
        CONFIG.figure_dir / "diagnostics",
        CONFIG.model_dir,
        CONFIG.metric_dir,
        CONFIG.prediction_dir,
        CONFIG.log_dir,
    ]:
        p.mkdir(parents=True, exist_ok=True)


def set_seed(seed=308):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


mkdirs()
set_seed(CONFIG.seed)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("Python       =", sys.executable)
print("RAW_CSV      =", CONFIG.raw_csv, CONFIG.raw_csv.exists())
print("DEVICE       =", CONFIG.device)
print("OUTPUT_ROOT  =", CONFIG.output_root)
if "transformers" not in sys.executable:
    print("提示：AGENTS.md 建议优先使用 conda 的 transformers 环境运行训练/推理。")


PROJECT_ROOT = /data2/hjs/pythonProject/pythonProject/Project4_wangluoliuliang
Python       = /home/hjs/anaconda3/envs/transformers/bin/python
RAW_CSV      = /data2/hjs/pythonProject/pythonProject/Project4_wangluoliuliang/raw_data.csv True
DEVICE       = cuda
OUTPUT_ROOT  = /data2/hjs/pythonProject/pythonProject/Project4_wangluoliuliang/outputs/network_traffic_forecast


## 2. 数据读取与无泄露预处理


In [3]:
def read_raw_csv(path):
    if not Path(path).exists():
        raise FileNotFoundError(f"missing raw csv: {path}")
    df = pd.read_csv(path)
    if CONFIG.time_col not in df.columns:
        raise ValueError(f"missing time column: {CONFIG.time_col}")
    if CONFIG.target_col not in df.columns:
        raise ValueError(f"missing target column: {CONFIG.target_col}")

    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    if df[[CONFIG.time_col, CONFIG.target_col]].isna().any().any():
        bad = df[[CONFIG.time_col, CONFIG.target_col]].isna().sum().to_dict()
        raise ValueError(f"time/target contains NaN after numeric conversion: {bad}")

    if df[CONFIG.time_col].duplicated().any():
        dup = int(df[CONFIG.time_col].duplicated().sum())
        raise ValueError(f"{CONFIG.time_col} contains duplicate time buckets: {dup}")

    df = df.sort_values(CONFIG.time_col).reset_index(drop=True)
    if not np.allclose(df[CONFIG.time_col], np.round(df[CONFIG.time_col])):
        raise ValueError(f"{CONFIG.time_col} must be integer-like contiguous bucket id")
    df[CONFIG.time_col] = df[CONFIG.time_col].astype(np.int64)

    diffs = df[CONFIG.time_col].diff().dropna()
    if not (diffs == 1).all():
        bad = diffs[diffs != 1].head().tolist()
        raise ValueError(f"{CONFIG.time_col} must be contiguous with step=1; first bad diffs={bad}")

    if df[CONFIG.target_col].lt(0).any():
        raise ValueError(f"{CONFIG.target_col} contains negative values, cannot use log1p transform")

    df["step"] = df[CONFIG.time_col].astype(np.int64)
    # 原始 CSV 没有绝对时间戳，这里只构造固定 10min 网格，便于画图和窗口索引。
    df["synthetic_time"] = pd.Timestamp("2000-01-01") + pd.to_timedelta(
        df["step"] * CONFIG.freq_minutes, unit="min"
    )
    df["hour"] = ((df["step"] * CONFIG.freq_minutes) // 60) % 24
    df["day_index"] = (df["step"] * CONFIG.freq_minutes) // (24 * 60)
    return df


def split_by_time(df):
    n = len(df)
    n_train = int(n * CONFIG.train_ratio)
    n_val = int(n * CONFIG.val_ratio)
    train = df.iloc[:n_train].copy()
    val = df.iloc[n_train:n_train + n_val].copy()
    test = df.iloc[n_train + n_val:].copy()
    if min(len(train), len(val), len(test)) <= CONFIG.input_len + CONFIG.pred_len:
        raise ValueError("split is too short for current input_len + pred_len")
    return {"train": train, "val": val, "test": test}


def standardize_splits(splits):
    train_log = np.log1p(splits["train"][CONFIG.target_col].to_numpy(dtype=np.float64))
    mean = float(train_log.mean())
    std = float(train_log.std(ddof=0))
    if std <= 0:
        std = 1.0

    out = {}
    for split_name, part in splits.items():
        part = part.copy()
        log_target = np.log1p(part[CONFIG.target_col].to_numpy(dtype=np.float64))
        part[f"{CONFIG.target_col}_log1p"] = log_target
        part[f"{CONFIG.target_col}_norm"] = (log_target - mean) / std
        part["split"] = split_name
        out[split_name] = part
    return out, {"target_log1p_mean_train": mean, "target_log1p_std_train": std}


def save_processed_splits(splits, stats, raw_rows, raw_columns):
    CONFIG.processed_dir.mkdir(parents=True, exist_ok=True)
    for split_name, part in splits.items():
        part.to_csv(CONFIG.processed_dir / f"{split_name}.csv", index=False)

    summary = {
        "raw_csv": str(CONFIG.raw_csv),
        "raw_rows": int(raw_rows),
        "raw_columns": list(raw_columns),
        "time_col": CONFIG.time_col,
        "time_col_meaning": "contiguous integer time bucket id; raw CSV does not contain absolute timestamps",
        "frequency": CONFIG.freq,
        "frequency_minutes": CONFIG.freq_minutes,
        "target_col": CONFIG.target_col,
        "target_transform": "log1p then train-split z-score",
        "model_input": "past normalized n_flows only; raw auxiliary network features are used for EDA, not for future-leaking training input",
        "input_len": CONFIG.input_len,
        "pred_len": CONFIG.pred_len,
        "input_horizon_hours": CONFIG.input_len * CONFIG.freq_minutes / 60,
        "prediction_horizon_hours": CONFIG.pred_len * CONFIG.freq_minutes / 60,
        "stride": CONFIG.stride,
        "split_ratio": {
            "train": CONFIG.train_ratio,
            "val": CONFIG.val_ratio,
            "test": 1 - CONFIG.train_ratio - CONFIG.val_ratio,
        },
        "split_rows": {name: int(len(part)) for name, part in splits.items()},
        **stats,
    }
    (CONFIG.processed_dir / "summary.json").write_text(
        json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    return summary


def make_windows(values, steps, times):
    total_len = CONFIG.input_len + CONFIG.pred_len
    xs, ys, rows = [], [], []
    for start in range(0, len(values) - total_len + 1, CONFIG.stride):
        enc_slice = slice(start, start + CONFIG.input_len)
        y_slice = slice(start + CONFIG.input_len, start + total_len)
        xs.append(values[enc_slice, None])
        ys.append(values[y_slice])
        rows.append({
            "input_start_step": int(steps[enc_slice][0]),
            "input_end_step": int(steps[enc_slice][-1]),
            "target_start_step": int(steps[y_slice][0]),
            "target_end_step": int(steps[y_slice][-1]),
            "input_start_time": str(times[enc_slice][0]),
            "input_end_time": str(times[enc_slice][-1]),
            "target_start_time": str(times[y_slice][0]),
            "target_end_time": str(times[y_slice][-1]),
        })
    if not xs:
        raise ValueError("not enough rows for one forecasting window")
    return np.asarray(xs, dtype=np.float32), np.asarray(ys, dtype=np.float32), pd.DataFrame(rows)


def save_window_splits(splits, stats):
    CONFIG.window_dir.mkdir(parents=True, exist_ok=True)
    summary = {
        "source_processed_dir": str(CONFIG.processed_dir),
        "output_dir": str(CONFIG.window_dir),
        "target_col": CONFIG.target_col,
        "target_transform": "log1p then train-split z-score",
        "frequency": CONFIG.freq,
        "frequency_minutes": CONFIG.freq_minutes,
        "input_len": CONFIG.input_len,
        "pred_len": CONFIG.pred_len,
        "input_horizon_hours": CONFIG.input_len * CONFIG.freq_minutes / 60,
        "prediction_horizon_hours": CONFIG.pred_len * CONFIG.freq_minutes / 60,
        "stride": CONFIG.stride,
        "schema": {
            "x": "past 168 steps of normalized n_flows only; shape (N, 168, 1)",
            "y": "future 72 steps of normalized n_flows; shape (N, 72)",
        },
        **stats,
        "splits": {},
    }

    for split_name, part in splits.items():
        values = part[f"{CONFIG.target_col}_norm"].to_numpy(dtype=np.float32)
        steps = part["step"].to_numpy()
        times = part["synthetic_time"].to_numpy()
        x, y, index_df = make_windows(values, steps, times)
        np.savez_compressed(
            CONFIG.window_dir / f"{split_name}.npz",
            x=x,
            y=y,
            target_col=np.array([CONFIG.target_col]),
            input_len=np.array([CONFIG.input_len], dtype=np.int64),
            pred_len=np.array([CONFIG.pred_len], dtype=np.int64),
            frequency=np.array([CONFIG.freq]),
            frequency_minutes=np.array([CONFIG.freq_minutes], dtype=np.int64),
            target_log1p_mean_train=np.array([stats["target_log1p_mean_train"]], dtype=np.float32),
            target_log1p_std_train=np.array([stats["target_log1p_std_train"]], dtype=np.float32),
        )
        index_df.to_csv(CONFIG.window_dir / f"{split_name}_index.csv", index=False)
        summary["splits"][split_name] = {
            "rows": int(len(part)),
            "samples": int(x.shape[0]),
            "x_shape": list(x.shape),
            "y_shape": list(y.shape),
            "first_window": index_df.iloc[0].to_dict(),
            "last_window": index_df.iloc[-1].to_dict(),
        }

    (CONFIG.window_dir / "summary.json").write_text(
        json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    return summary


def load_processed_splits():
    return {
        name: pd.read_csv(CONFIG.processed_dir / f"{name}.csv", parse_dates=["synthetic_time"])
        for name in ["train", "val", "test"]
    }


def prepare_forecast_data(force_rebuild=False):
    required = [
        CONFIG.processed_dir / "summary.json",
        CONFIG.window_dir / "summary.json",
        CONFIG.window_dir / "train.npz",
        CONFIG.window_dir / "val.npz",
        CONFIG.window_dir / "test.npz",
    ]
    if (not force_rebuild) and all(p.exists() for p in required):
        processed_summary = json.loads((CONFIG.processed_dir / "summary.json").read_text(encoding="utf-8"))
        window_summary = json.loads((CONFIG.window_dir / "summary.json").read_text(encoding="utf-8"))
        splits = load_processed_splits()
        print("复用已存在的预处理数据和窗口样本。")
        return {"processed_summary": processed_summary, "window_summary": window_summary, "splits": splits}

    raw_df = read_raw_csv(CONFIG.raw_csv)
    splits = split_by_time(raw_df)
    splits, stats = standardize_splits(splits)
    processed_summary = save_processed_splits(splits, stats, raw_rows=len(raw_df), raw_columns=raw_df.columns)
    window_summary = save_window_splits(splits, stats)
    print("已重新生成预处理数据和窗口样本。")
    return {"processed_summary": processed_summary, "window_summary": window_summary, "splits": splits}


## 3. Dataset / DataLoader 与基础可视化


In [4]:
class ForecastDataset(Dataset):
    def __init__(self, x, y):
        self.x = x.astype(np.float32)
        self.y = y.astype(np.float32)

    def __len__(self):
        return self.x.shape[0]

    def __getitem__(self, idx):
        return torch.from_numpy(self.x[idx]), torch.from_numpy(self.y[idx]), idx


def load_npz_split(split_name):
    data = np.load(CONFIG.window_dir / f"{split_name}.npz", allow_pickle=True)
    return {k: data[k] for k in data.files}


def trim_split_data(data, max_samples=None):
    if max_samples is None:
        return data
    out = dict(data)
    n = min(max_samples, out["x"].shape[0])
    out["x"] = out["x"][:n]
    out["y"] = out["y"][:n]
    return out


def build_loader(split_data, batch_size, shuffle):
    ds = ForecastDataset(split_data["x"], split_data["y"])
    loader_kwargs = {
        "batch_size": batch_size,
        "shuffle": shuffle,
        "num_workers": CONFIG.n_workers,
        "pin_memory": torch.cuda.is_available(),
    }
    if CONFIG.n_workers > 0:
        loader_kwargs["persistent_workers"] = True
        loader_kwargs["prefetch_factor"] = 4
    return ds, DataLoader(ds, **loader_kwargs)


def load_train_val_test(max_samples=None):
    train_data = trim_split_data(load_npz_split("train"), max_samples)
    val_data = trim_split_data(load_npz_split("val"), max_samples)
    test_data = trim_split_data(load_npz_split("test"), max_samples)
    return train_data, val_data, test_data


def inverse_target(z, mean, std):
    return np.expm1(z * std + mean).clip(min=0.0)


def regression_metrics(y_true, y_pred):
    err = y_pred - y_true
    return {
        "MAE": float(np.mean(np.abs(err))),
        "MSE": float(np.mean(err ** 2)),
        "RMSE": float(np.sqrt(np.mean(err ** 2))),
        "MAPE(%)": float(np.mean(np.abs(err) / np.clip(np.abs(y_true), 1e-6, None)) * 100),
    }


def savefig(path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()


def plot_corr_heatmap(df, out_path):
    num_df = df.select_dtypes(include=[np.number]).copy()
    cols = [c for c in num_df.columns if c not in {"date", "step", "hour", "day_index"}]
    corr = num_df[cols].corr()
    plt.figure(figsize=(10, 8))
    if sns is not None:
        sns.heatmap(corr, cmap="RdBu_r", center=0, square=True, linewidths=0.2)
    else:
        plt.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
        plt.xticks(range(len(corr.columns)), corr.columns, rotation=75, ha="right")
        plt.yticks(range(len(corr.index)), corr.index)
        plt.colorbar()
    plt.title("Network Feature Correlation Heatmap")
    savefig(out_path)


def plot_acf(values, out_path, max_lag=288):
    values = np.asarray(values, dtype=np.float64)
    values = values - values.mean()
    denom = np.sum(values ** 2)
    lags, acfs = [], []
    for lag in range(1, min(max_lag, len(values) - 1) + 1):
        acf = float(np.sum(values[:-lag] * values[lag:]) / max(denom, 1e-12))
        lags.append(lag)
        acfs.append(acf)
    plt.figure(figsize=(10, 4))
    plt.plot(np.asarray(lags) * CONFIG.freq_minutes / 60, acfs, linewidth=1.6)
    plt.axhline(0, color="black", linewidth=0.8)
    plt.xlabel("Lag (hours)")
    plt.ylabel("ACF")
    plt.title("Target Autocorrelation")
    plt.grid(alpha=0.25)
    savefig(out_path)


def plot_spectrum(values, out_path):
    values = np.asarray(values, dtype=np.float64)
    values = values - values.mean()
    fft = np.fft.rfft(values)
    power = np.abs(fft) ** 2
    freq = np.fft.rfftfreq(len(values), d=CONFIG.freq_minutes / 60)
    keep = freq > 0
    plt.figure(figsize=(10, 4))
    plt.plot(freq[keep], power[keep], linewidth=1.2)
    plt.xlim(0, np.percentile(freq[keep], 85))
    plt.xlabel("Frequency (cycles/hour)")
    plt.ylabel("Power")
    plt.title("Target Frequency Spectrum")
    plt.grid(alpha=0.25)
    savefig(out_path)


def plot_data_visualizations(data_pack):
    fig_dir = CONFIG.figure_dir / "data"
    raw_df = read_raw_csv(CONFIG.raw_csv)
    split_df = pd.concat(data_pack["splits"].values(), ignore_index=True)

    plt.figure(figsize=(14, 4.2))
    for split_name, part in data_pack["splits"].items():
        plt.plot(part["synthetic_time"], part[CONFIG.target_col], label=split_name, linewidth=1.0)
    plt.xlabel("Synthetic Time")
    plt.ylabel(CONFIG.target_col)
    plt.title("Network Traffic Timeline with Train / Val / Test Split")
    plt.legend()
    plt.grid(alpha=0.25)
    savefig(fig_dir / "01_target_timeline_split.png")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].hist(raw_df[CONFIG.target_col], bins=80, color="#4C78A8", alpha=0.85)
    axes[0].set_title("Raw n_flows Distribution")
    axes[1].hist(np.log1p(raw_df[CONFIG.target_col]), bins=80, color="#72B7B2", alpha=0.85)
    axes[1].set_title("log1p(n_flows) Distribution")
    axes[2].hist(split_df[f"{CONFIG.target_col}_norm"], bins=80, color="#F58518", alpha=0.85)
    axes[2].set_title("Train-Standardized Target Distribution")
    for ax in axes:
        ax.grid(alpha=0.25)
    savefig(fig_dir / "02_target_distribution_transform.png")

    train = data_pack["splits"]["train"].copy()
    roll_mean = train[CONFIG.target_col].rolling(144, min_periods=10).mean()
    roll_std = train[CONFIG.target_col].rolling(144, min_periods=10).std()
    plt.figure(figsize=(14, 4.2))
    plt.plot(train["synthetic_time"], train[CONFIG.target_col], label="train target", linewidth=0.7, alpha=0.45)
    plt.plot(train["synthetic_time"], roll_mean, label="24h rolling mean", linewidth=1.8)
    plt.plot(train["synthetic_time"], roll_std, label="24h rolling std", linewidth=1.5)
    plt.xlabel("Synthetic Time")
    plt.ylabel(CONFIG.target_col)
    plt.title("Train Target Rolling Mean / Std")
    plt.legend()
    plt.grid(alpha=0.25)
    savefig(fig_dir / "03_train_rolling_mean_std.png")

    plot_corr_heatmap(raw_df, fig_dir / "04_raw_feature_correlation_heatmap.png")

    hourly = raw_df.groupby("hour")[CONFIG.target_col].agg(["mean", "median", "std"]).reset_index()
    plt.figure(figsize=(10, 4.2))
    plt.plot(hourly["hour"], hourly["mean"], marker="o", label="mean")
    plt.plot(hourly["hour"], hourly["median"], marker="s", label="median")
    plt.fill_between(hourly["hour"], hourly["mean"] - hourly["std"], hourly["mean"] + hourly["std"], alpha=0.18)
    plt.xlabel("Hour of Day")
    plt.ylabel(CONFIG.target_col)
    plt.title("Intraday Traffic Pattern")
    plt.xticks(range(0, 24, 2))
    plt.legend()
    plt.grid(alpha=0.25)
    savefig(fig_dir / "05_hourly_traffic_pattern.png")

    plot_acf(train[CONFIG.target_col].values, fig_dir / "06_target_autocorrelation.png")
    plot_spectrum(train[CONFIG.target_col].values, fig_dir / "07_target_frequency_spectrum.png")

    rows = data_pack["processed_summary"]["split_rows"]
    samples = {k: v["samples"] for k, v in data_pack["window_summary"]["splits"].items()}
    x = np.arange(len(rows))
    plt.figure(figsize=(8, 4.2))
    plt.bar(x - 0.18, list(rows.values()), width=0.36, label="rows", color="#4C78A8")
    plt.bar(x + 0.18, list(samples.values()), width=0.36, label="windows", color="#F58518")
    plt.xticks(x, list(rows.keys()))
    plt.ylabel("Count")
    plt.title("Split Rows and Window Samples")
    plt.legend()
    plt.grid(axis="y", alpha=0.25)
    savefig(fig_dir / "08_split_rows_and_windows.png")

    raw_numeric = raw_df.select_dtypes(include=[np.number]).drop(columns=["date", "step", "hour", "day_index"], errors="ignore")
    corr_target = raw_numeric.corr()[CONFIG.target_col].drop(CONFIG.target_col).sort_values(key=lambda x: x.abs(), ascending=False).head(8)
    plt.figure(figsize=(10, 4.8))
    colors = ["#4C78A8" if v >= 0 else "#E45756" for v in corr_target.values]
    x = np.arange(len(corr_target))
    plt.bar(x, corr_target.values, color=colors, alpha=0.88)
    plt.axhline(0, color="black", linewidth=0.8)
    plt.xticks(x, corr_target.index, rotation=35, ha="right")
    plt.ylabel("Pearson Correlation with n_flows")
    plt.title("Top Raw Feature Correlations for EDA")
    plt.grid(axis="y", alpha=0.25)
    savefig(fig_dir / "09_top_feature_target_correlation.png")

    print(f"数据可视化已保存到: {fig_dir}")


## 4. 模型定义：Baseline、PatchTST 与 SDF-PatchTST


In [5]:
class MovingAverage(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        if kernel_size <= 0:
            raise ValueError("kernel_size must be positive")
        self.kernel_size = kernel_size
        self.left_pad = kernel_size - 1

    def forward(self, x):
        x_pad = F.pad(x.transpose(1, 2), (self.left_pad, 0), mode="replicate")
        trend = F.avg_pool1d(x_pad, kernel_size=self.kernel_size, stride=1)
        return trend.transpose(1, 2)


class DLinearModel(nn.Module):
    def __init__(self, input_len, pred_len, ma_kernel):
        super().__init__()
        self.ma = MovingAverage(ma_kernel)
        self.trend_head = nn.Linear(input_len, pred_len)
        self.seasonal_head = nn.Linear(input_len, pred_len)

    def forward(self, x):
        trend = self.ma(x)
        seasonal = x - trend
        return self.trend_head(trend.squeeze(-1)) + self.seasonal_head(seasonal.squeeze(-1))


class LearnablePositionalEncoding(nn.Module):
    def __init__(self, seq_len, d_model):
        super().__init__()
        self.pos = nn.Parameter(torch.zeros(1, seq_len, d_model))
        nn.init.trunc_normal_(self.pos, std=0.02)

    def forward(self, x):
        return x + self.pos[:, :x.size(1), :]


class TransformerModel(nn.Module):
    def __init__(self, input_len, pred_len, d_model, n_heads, layers, d_ff, dropout):
        super().__init__()
        self.in_proj = nn.Linear(1, d_model)
        self.pos = LearnablePositionalEncoding(input_len, d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.Flatten(start_dim=1),
            nn.Dropout(dropout),
            nn.Linear(input_len * d_model, pred_len),
        )

    def forward(self, x):
        z = self.pos(self.in_proj(x))
        z = self.encoder(z)
        return self.head(self.norm(z))


class AutoCorrelationLite(nn.Module):
    def __init__(self, d_model, dropout):
        super().__init__()
        self.value = nn.Linear(d_model, d_model)
        self.out = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, seq_len, width = x.shape
        value = self.value(x)
        signal = x.mean(dim=-1)
        spectrum = torch.fft.rfft(signal, dim=1)
        corr = torch.fft.irfft(spectrum * torch.conj(spectrum), n=seq_len, dim=1)
        top_k = max(1, min(seq_len, int(math.log(seq_len + 1))))
        weights, delays = torch.topk(corr, k=top_k, dim=1)
        weights = torch.softmax(weights, dim=1)

        time_index = torch.arange(seq_len, device=x.device).view(1, 1, seq_len)
        gather_index = (time_index + delays.unsqueeze(-1)) % seq_len
        gather_index = gather_index.unsqueeze(-1).expand(batch_size, top_k, seq_len, width)
        rolled = torch.gather(
            value.unsqueeze(1).expand(batch_size, top_k, seq_len, width),
            dim=2,
            index=gather_index,
        )
        mixed = (weights.view(batch_size, top_k, 1, 1) * rolled).sum(dim=1)
        return self.out(self.dropout(mixed))


class AutoformerBlock(nn.Module):
    def __init__(self, d_model, d_ff, ma_kernel, dropout):
        super().__init__()
        self.auto_corr = AutoCorrelationLite(d_model, dropout)
        self.decomp1 = MovingAverage(ma_kernel)
        self.decomp2 = MovingAverage(ma_kernel)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.dropout(self.auto_corr(self.norm1(x)))
        x = x - self.decomp1(x)
        x = x + self.dropout(self.ffn(self.norm2(x)))
        return x - self.decomp2(x)


class AutoformerModel(nn.Module):
    def __init__(self, input_len, pred_len, d_model, layers, d_ff, ma_kernel, dropout):
        super().__init__()
        self.ma = MovingAverage(ma_kernel)
        self.trend_head = nn.Linear(input_len, pred_len)
        self.in_proj = nn.Linear(1, d_model)
        self.pos = LearnablePositionalEncoding(input_len, d_model)
        self.blocks = nn.ModuleList([
            AutoformerBlock(d_model, d_ff, ma_kernel, dropout)
            for _ in range(layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.seasonal_head = nn.Sequential(
            nn.Flatten(start_dim=1),
            nn.Dropout(dropout),
            nn.Linear(input_len * d_model, pred_len),
        )

    def forward(self, x):
        trend = self.ma(x)
        seasonal = x - trend
        z = self.pos(self.in_proj(seasonal))
        for block in self.blocks:
            z = block(z)
        return self.seasonal_head(self.norm(z)) + self.trend_head(trend.squeeze(-1))


class TimeXerPatchEmbedding(nn.Module):
    def __init__(self, input_len, patch_len, stride, d_model, dropout):
        super().__init__()
        if patch_len > input_len:
            raise ValueError(f"patch_len={patch_len} exceeds input_len={input_len}")
        if stride <= 0:
            raise ValueError("stride must be positive")
        self.patch_len = patch_len
        self.stride = stride
        self.n_patch = (input_len - patch_len) // stride + 1
        self.proj = nn.Linear(patch_len, d_model)
        self.pos = nn.Parameter(torch.zeros(1, self.n_patch, d_model))
        self.dropout = nn.Dropout(dropout)
        nn.init.trunc_normal_(self.pos, std=0.02)

    def forward(self, x):
        patches = x.squeeze(-1).unfold(dimension=1, size=self.patch_len, step=self.stride)
        return self.dropout(self.proj(patches) + self.pos)


class TimeXerModel(nn.Module):
    def __init__(self, input_len, pred_len, patch_len, stride, d_model, n_heads, layers, d_ff, dropout):
        super().__init__()
        self.patch = TimeXerPatchEmbedding(input_len, patch_len, stride, d_model, dropout)
        self.global_token = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.trunc_normal_(self.global_token, std=0.02)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.Flatten(start_dim=1),
            nn.Dropout(dropout),
            nn.Linear((self.patch.n_patch + 1) * d_model, pred_len),
        )

    def forward(self, x):
        patches = self.patch(x)
        global_token = self.global_token.expand(x.size(0), -1, -1)
        z = self.encoder(torch.cat([global_token, patches], dim=1))
        return self.head(self.norm(z))


class FrequencyMixingBlock(nn.Module):
    def __init__(self, seq_len, modes):
        super().__init__()
        self.seq_len = seq_len
        self.n_freq = seq_len // 2 + 1
        self.modes = max(1, min(modes, self.n_freq))
        self.weight_real = nn.Parameter(torch.randn(self.modes) * 0.02)
        self.weight_imag = nn.Parameter(torch.randn(self.modes) * 0.02)

    def forward(self, x):
        xf = torch.fft.rfft(x.squeeze(-1), dim=1)
        out = torch.zeros_like(xf)
        weight = torch.complex(self.weight_real, self.weight_imag).view(1, -1)
        out[:, :self.modes] = xf[:, :self.modes] * weight
        return torch.fft.irfft(out, n=self.seq_len, dim=1).unsqueeze(-1)


class FEDformerModel(nn.Module):
    def __init__(self, input_len, pred_len, ma_kernel, modes):
        super().__init__()
        self.ma = MovingAverage(ma_kernel)
        self.freq = FrequencyMixingBlock(input_len, modes)
        self.trend_head = nn.Linear(input_len, pred_len)
        self.seasonal_head = nn.Linear(input_len, pred_len)

    def forward(self, x):
        trend = self.ma(x)
        seasonal = self.freq(x - trend)
        return self.trend_head(trend.squeeze(-1)) + self.seasonal_head(seasonal.squeeze(-1))


class InvertedEncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        z = self.norm1(x)
        attn, _ = self.attn(z, z, z, need_weights=False)
        x = x + self.dropout(attn)
        return x + self.dropout(self.ffn(self.norm2(x)))


class iTransformerModel(nn.Module):
    def __init__(self, input_len, pred_len, d_model, n_heads, layers, d_ff, dropout):
        super().__init__()
        self.value_proj = nn.Linear(input_len, d_model)
        self.blocks = nn.ModuleList([
            InvertedEncoderBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, pred_len)

    def forward(self, x):
        z = self.value_proj(x.transpose(1, 2))
        for block in self.blocks:
            z = block(z)
        return self.head(self.norm(z)).squeeze(1)


class RevIN(nn.Module):
    def __init__(self, num_features, eps=1e-5, affine=True):
        super().__init__()
        self.eps = eps
        self.affine = affine
        if affine:
            self.gamma = nn.Parameter(torch.ones(1, 1, num_features))
            self.beta = nn.Parameter(torch.zeros(1, 1, num_features))
        self._cached_mean = None
        self._cached_std = None

    def forward(self, x, mode):
        if mode == "norm":
            self._cached_mean = x.mean(dim=1, keepdim=True)
            self._cached_std = x.std(dim=1, keepdim=True, unbiased=False).clamp_min(self.eps)
            out = (x - self._cached_mean) / self._cached_std
            if self.affine:
                out = out * self.gamma + self.beta
            return out
        if mode == "denorm":
            out = x
            if self.affine:
                out = (out - self.beta) / self.gamma.clamp_min(self.eps)
            return out * self._cached_std + self._cached_mean
        raise ValueError(f"Unknown RevIN mode: {mode}")


class PatchEmbedding(nn.Module):
    def __init__(self, seq_len, patch_len, stride, d_model, dropout):
        super().__init__()
        if patch_len > seq_len:
            raise ValueError(f"patch_len={patch_len} must be <= seq_len={seq_len}")
        if stride <= 0:
            raise ValueError("stride must be positive")
        self.patch_len = patch_len
        self.stride = stride
        self.n_patches = (seq_len - patch_len) // stride + 1
        self.proj = nn.Linear(patch_len, d_model)
        self.pos_embed = nn.Parameter(torch.zeros(1, self.n_patches, d_model))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        patches = x.unfold(dimension=-1, size=self.patch_len, step=self.stride)
        bsz, nvars, n_patch, p_len = patches.shape
        tokens = patches.reshape(bsz * nvars, n_patch, p_len)
        tokens = self.proj(tokens) + self.pos_embed[:, :n_patch, :]
        return self.dropout(tokens)


class LearnableSpectralFilter(nn.Module):
    def __init__(self, input_len, c_in):
        super().__init__()
        n_freq = input_len // 2 + 1
        init = math.log(math.exp(1.0) - 1.0)
        self.log_mask = nn.Parameter(torch.full((1, n_freq, c_in), init))

    def forward(self, x):
        _, t, _ = x.shape
        mask = F.softplus(self.log_mask)
        x_freq = torch.fft.rfft(x, dim=1)
        x_freq = x_freq * mask
        return torch.fft.irfft(x_freq, n=t, dim=1)


class CausalSeriesDecomp(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        if kernel_size <= 0:
            raise ValueError("ma_kernel must be positive")
        self.kernel_size = kernel_size
        self.padding = kernel_size - 1

    def forward(self, x):
        # 因果移动平均只看当前和过去，不把未来点混入历史窗口。
        x_pad = F.pad(x.transpose(1, 2), (self.padding, 0), mode="replicate")
        trend = F.avg_pool1d(x_pad, kernel_size=self.kernel_size, stride=1).transpose(1, 2)
        seasonal = x - trend
        return seasonal, trend


class TrendSeasonalPatchEmbedding(nn.Module):
    def __init__(self, seq_len, patch_len, stride, d_model, dropout, ma_kernel):
        super().__init__()
        if patch_len > seq_len:
            raise ValueError(f"patch_len={patch_len} must be <= seq_len={seq_len}")
        if stride <= 0:
            raise ValueError("stride must be positive")
        self.patch_len = patch_len
        self.stride = stride
        self.n_patches = (seq_len - patch_len) // stride + 1
        self.decomp = CausalSeriesDecomp(ma_kernel)
        self.seasonal_proj = nn.Linear(patch_len, d_model)
        self.trend_proj = nn.Linear(patch_len, d_model)
        self.pos_embed = nn.Parameter(torch.zeros(1, self.n_patches, d_model))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def _patchify(self, x):
        x = x.permute(0, 2, 1)
        patches = x.unfold(dimension=-1, size=self.patch_len, step=self.stride)
        bsz, nvars, n_patch, p_len = patches.shape
        return patches.reshape(bsz * nvars, n_patch, p_len)

    def forward(self, x):
        seasonal, trend = self.decomp(x)
        seasonal_tokens = self.seasonal_proj(self._patchify(seasonal))
        trend_tokens = self.trend_proj(self._patchify(trend))
        tokens = seasonal_tokens + trend_tokens
        tokens = tokens + self.pos_embed[:, :tokens.size(1), :]
        return self.dropout(self.norm(tokens))


class PatchTSTModel(nn.Module):
    def __init__(
        self,
        seq_len,
        pred_len,
        c_in,
        patch_len,
        stride,
        d_model,
        n_heads,
        e_layers,
        d_ff,
        dropout,
        use_revin=False,
        revin_affine=True,
    ):
        super().__init__()
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.c_in = c_in
        self.n_patches = (seq_len - patch_len) // stride + 1
        self.use_revin = use_revin
        self.revin = RevIN(c_in, affine=revin_affine) if use_revin else None
        self.patch_embed = PatchEmbedding(seq_len, patch_len, stride, d_model, dropout)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=e_layers)
        self.encoder_norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.Flatten(start_dim=1),
            nn.Dropout(dropout),
            nn.Linear(c_in * self.n_patches * d_model, pred_len),
        )

    def forward(self, x):
        if self.use_revin:
            x = self.revin(x, "norm")
        bsz, _, nvars = x.shape
        tokens = self.patch_embed(x)
        tokens = self.encoder(tokens)
        tokens = self.encoder_norm(tokens)
        tokens = tokens.reshape(bsz, nvars, self.n_patches, -1)
        return self.head(tokens)


class SDFPatchTSTModel(nn.Module):
    def __init__(
        self,
        seq_len,
        pred_len,
        c_in,
        patch_len,
        stride,
        d_model,
        n_heads,
        e_layers,
        d_ff,
        dropout,
        ma_kernel,
        use_spectral=True,
        use_decomp=True,
        use_revin=False,
        revin_affine=True,
    ):
        super().__init__()
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.c_in = c_in
        self.n_patches = (seq_len - patch_len) // stride + 1
        self.use_revin = use_revin
        self.revin = RevIN(c_in, affine=revin_affine) if use_revin else None
        self.spectral_filter = LearnableSpectralFilter(seq_len, c_in) if use_spectral else nn.Identity()
        if use_decomp:
            self.patch_embed = TrendSeasonalPatchEmbedding(
                seq_len=seq_len,
                patch_len=patch_len,
                stride=stride,
                d_model=d_model,
                dropout=dropout,
                ma_kernel=ma_kernel,
            )
        else:
            self.patch_embed = PatchEmbedding(seq_len, patch_len, stride, d_model, dropout)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=e_layers)
        self.encoder_norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.Flatten(start_dim=1),
            nn.Dropout(dropout),
            nn.Linear(c_in * self.n_patches * d_model, pred_len),
        )

    def forward(self, x):
        if self.use_revin:
            x = self.revin(x, "norm")
        x = self.spectral_filter(x)
        bsz, _, nvars = x.shape
        tokens = self.patch_embed(x)
        tokens = self.encoder(tokens)
        tokens = self.encoder_norm(tokens)
        tokens = tokens.reshape(bsz, nvars, self.n_patches, -1)
        return self.head(tokens)


def build_model(model_name, train_data):
    input_len = int(train_data["input_len"][0])
    pred_len = int(train_data["pred_len"][0])
    c_in = int(train_data["x"].shape[-1])
    if c_in != 1:
        raise ValueError("当前一键版默认单变量 n_flows 输入，c_in 必须为 1。")

    if model_name == "DLinear":
        return DLinearModel(input_len, pred_len, CONFIG.ma_kernel)
    if model_name == "Transformer":
        return TransformerModel(input_len, pred_len, CONFIG.d_model, CONFIG.n_heads, CONFIG.e_layers, CONFIG.d_ff, CONFIG.dropout)
    if model_name == "Autoformer":
        return AutoformerModel(input_len, pred_len, CONFIG.d_model, CONFIG.e_layers, CONFIG.d_ff, CONFIG.ma_kernel, CONFIG.dropout)
    if model_name == "TimeXer":
        return TimeXerModel(input_len, pred_len, CONFIG.patch_len, CONFIG.patch_stride, CONFIG.d_model, CONFIG.n_heads, CONFIG.e_layers, CONFIG.d_ff, CONFIG.dropout)
    if model_name == "FEDformer":
        return FEDformerModel(input_len, pred_len, CONFIG.ma_kernel, CONFIG.fed_modes)
    if model_name == "iTransformer":
        return iTransformerModel(input_len, pred_len, CONFIG.d_model, CONFIG.n_heads, CONFIG.e_layers, CONFIG.d_ff, CONFIG.dropout)
    if model_name == "PatchTST":
        return PatchTSTModel(input_len, pred_len, c_in, CONFIG.patch_len, CONFIG.patch_stride, CONFIG.d_model, CONFIG.n_heads, CONFIG.e_layers, CONFIG.d_ff, CONFIG.dropout, CONFIG.use_revin)
    if model_name == "SDF-PatchTST":
        return SDFPatchTSTModel(input_len, pred_len, c_in, CONFIG.patch_len, CONFIG.patch_stride, CONFIG.d_model, CONFIG.n_heads, CONFIG.e_layers, CONFIG.d_ff, CONFIG.dropout, CONFIG.ma_kernel, True, True, CONFIG.use_revin)
    if model_name == "SDF-PatchTST w/o Decomp":
        return SDFPatchTSTModel(input_len, pred_len, c_in, CONFIG.patch_len, CONFIG.patch_stride, CONFIG.d_model, CONFIG.n_heads, CONFIG.e_layers, CONFIG.d_ff, CONFIG.dropout, CONFIG.ma_kernel, True, False, CONFIG.use_revin)
    if model_name == "SDF-PatchTST w/o Spectral":
        return SDFPatchTSTModel(input_len, pred_len, c_in, CONFIG.patch_len, CONFIG.patch_stride, CONFIG.d_model, CONFIG.n_heads, CONFIG.e_layers, CONFIG.d_ff, CONFIG.dropout, CONFIG.ma_kernel, False, True, CONFIG.use_revin)
    raise ValueError(f"unknown model_name={model_name}")


## 5. 训练、评估与结果保存函数


In [6]:
@dataclass
class EarlyStopping:
    patience: int
    min_delta: float = 0.0
    best: float = float("inf")
    bad_epochs: int = 0

    def step(self, value):
        if value < self.best - self.min_delta:
            self.best = float(value)
            self.bad_epochs = 0
            return False, True
        self.bad_epochs += 1
        return self.bad_epochs >= self.patience, False


def safe_model_name(model_name):
    return (
        model_name.replace("/", "_")
        .replace(" ", "_")
        .replace("-", "_")
        .replace("(", "")
        .replace(")", "")
    )


def count_params(model):
    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))


def train_one_epoch(model, loader, criterion, optimizer, device, grad_clip, epoch, total_epochs, model_name):
    model.train()
    total_loss = 0.0
    total_n = 0
    bar = tqdm(loader, desc=f"{model_name} train {epoch}/{total_epochs}", leave=False)
    for x, y, _ in bar:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        pred = model(x)
        loss = criterion(pred, y)
        loss.backward()
        if grad_clip > 0:
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        bs = x.size(0)
        total_loss += loss.item() * bs
        total_n += bs
        bar.set_postfix(loss=f"{loss.item():.6f}")
    return total_loss / max(1, total_n)


@torch.no_grad()
def eval_one_epoch(model, loader, criterion, device, split_name, model_name):
    model.eval()
    total_loss = 0.0
    total_n = 0
    bar = tqdm(loader, desc=f"{model_name} {split_name}", leave=False)
    for x, y, _ in bar:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        pred = model(x)
        loss = criterion(pred, y)
        bs = x.size(0)
        total_loss += loss.item() * bs
        total_n += bs
        bar.set_postfix(loss=f"{loss.item():.6f}")
    return total_loss / max(1, total_n)


@torch.no_grad()
def predict_all(model, loader, device):
    model.eval()
    preds, trues, idxs = [], [], []
    for x, y, idx in loader:
        pred = model(x.to(device, non_blocking=True)).cpu().numpy()
        preds.append(pred)
        trues.append(y.numpy())
        idxs.append(idx.numpy())
    order = np.argsort(np.concatenate(idxs))
    return np.concatenate(preds, axis=0)[order], np.concatenate(trues, axis=0)[order]


def plot_training_curve(history, out_path, title):
    plt.figure(figsize=(8, 4))
    plt.plot(history.get("train_loss", []), label="Train MSE")
    plt.plot(history.get("val_loss", []), label="Val MSE")
    plt.xlabel("Epoch")
    plt.ylabel("MSE")
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.25)
    savefig(out_path)


def plot_random_samples(x_norm, y_true_norm, y_pred_norm, mean, std, index_df, out_path, n_samples=6, title_prefix=""):
    n = min(n_samples, y_true_norm.shape[0])
    if n <= 0:
        return
    rng = np.random.default_rng(CONFIG.seed)
    ids = rng.choice(y_true_norm.shape[0], size=n, replace=False)
    fig, axes = plt.subplots(n, 1, figsize=(10, 3.4 * n), squeeze=False)
    for ax, sid in zip(axes[:, 0], ids):
        hist = inverse_target(x_norm[sid, :, 0], mean, std)
        true = inverse_target(y_true_norm[sid], mean, std)
        pred = inverse_target(y_pred_norm[sid], mean, std)
        ax.plot(np.arange(-len(hist) + 1, 1), hist, label="History", linewidth=1.6)
        ax.plot(np.arange(1, len(true) + 1), true, label="True", linewidth=1.8)
        ax.plot(np.arange(1, len(pred) + 1), pred, label="Pred", linewidth=1.8)
        ax.axvline(0, color="gray", linestyle="--", linewidth=1)
        plot_title = f"{title_prefix} Sample #{sid}"
        if index_df is not None and sid < len(index_df):
            plot_title += f" | target_start_step={index_df.iloc[sid]['target_start_step']}"
        ax.set_title(plot_title)
        ax.set_xlabel("10-minute step")
        ax.set_ylabel(CONFIG.target_col)
        ax.grid(alpha=0.25)
        ax.legend(loc="best")
    savefig(out_path)


def run_one_model(model_name, run_idx, max_samples=None):
    set_seed(CONFIG.seed + run_idx)
    device = torch.device(CONFIG.device)
    train_data, val_data, test_data = load_train_val_test(max_samples=max_samples)
    train_ds, train_loader = build_loader(train_data, CONFIG.batch_size, True)
    _, val_loader = build_loader(val_data, CONFIG.batch_size, False)
    test_ds, test_loader = build_loader(test_data, CONFIG.batch_size, False)

    model = build_model(model_name, train_data).to(device)
    n_params = count_params(model)
    criterion = nn.MSELoss()
    safe_name = safe_model_name(model_name)
    out_dir = CONFIG.run_dir / safe_name / f"run_{run_idx:02d}"
    out_dir.mkdir(parents=True, exist_ok=True)
    log_lines = []

    input_len = int(train_data["input_len"][0])
    pred_len = int(train_data["pred_len"][0])
    freq_minutes = int(train_data["frequency_minutes"][0])
    mean = float(train_data["target_log1p_mean_train"][0])
    std = float(train_data["target_log1p_std_train"][0])

    print(f"\\n{'=' * 80}\\n{model_name} | run {run_idx:02d}\\n{'=' * 80}")
    print(f"params={n_params:,} | train={len(train_ds)} | val={val_data['x'].shape[0]} | test={test_data['x'].shape[0]}")
    print(f"input={input_len} steps ({input_len * freq_minutes / 60:.1f}h), pred={pred_len} steps ({pred_len * freq_minutes / 60:.1f}h)")

    history = defaultdict(list)
    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    best_epoch = 0

    if n_params == 0:
        val_loss = eval_one_epoch(model, val_loader, criterion, device, "val", model_name)
        history["train_loss"].append(float("nan"))
        history["val_loss"].append(float(val_loss))
        best_val = float(val_loss)
        log_lines.append(f"parameter-free direct eval | val_mse={val_loss:.8f}")
    else:
        optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG.lr, weight_decay=CONFIG.weight_decay)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=CONFIG.lr_factor,
            patience=CONFIG.lr_patience,
            min_lr=CONFIG.min_lr,
        )
        early_stop = EarlyStopping(CONFIG.patience, CONFIG.min_delta)
        for epoch in range(1, CONFIG.epochs + 1):
            train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device, CONFIG.grad_clip, epoch, CONFIG.epochs, model_name)
            val_loss = eval_one_epoch(model, val_loader, criterion, device, "val", model_name)
            scheduler.step(val_loss)
            history["train_loss"].append(float(train_loss))
            history["val_loss"].append(float(val_loss))
            history["lr"].append(float(optimizer.param_groups[0]["lr"]))
            stop, improved = early_stop.step(val_loss)
            if improved:
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                best_epoch = epoch
            line = f"epoch={epoch:03d} train_mse={train_loss:.8f} val_mse={val_loss:.8f} lr={optimizer.param_groups[0]['lr']:.3e}"
            print(line)
            log_lines.append(line)
            if stop:
                stop_line = f"early_stop epoch={epoch:03d} best_epoch={best_epoch:03d} best_val_mse={early_stop.best:.8f}"
                print(stop_line)
                log_lines.append(stop_line)
                break
        best_val = float(early_stop.best)

    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    test_loss = eval_one_epoch(model, test_loader, criterion, device, "test", model_name)
    y_pred_norm, y_true_norm = predict_all(model, test_loader, device)
    y_pred_raw = inverse_target(y_pred_norm, mean, std)
    y_true_raw = inverse_target(y_true_norm, mean, std)
    norm_metrics = regression_metrics(y_true_norm, y_pred_norm)
    raw_metrics = regression_metrics(y_true_raw, y_pred_raw)

    torch.save(best_state, out_dir / "best_model.pt")
    np.save(out_dir / "test_pred_norm.npy", y_pred_norm)
    np.save(out_dir / "test_true_norm.npy", y_true_norm)
    np.save(out_dir / "test_pred_raw.npy", y_pred_raw)
    np.save(out_dir / "test_true_raw.npy", y_true_raw)
    (out_dir / "history.json").write_text(json.dumps(history, ensure_ascii=False, indent=2), encoding="utf-8")
    (out_dir / "train.log").write_text("\\n".join(log_lines) + "\\n", encoding="utf-8")

    plot_training_curve(history, CONFIG.figure_dir / "training" / f"{safe_name}_loss_curve.png", f"{model_name} Loss Curve")
    index_path = CONFIG.window_dir / "test_index.csv"
    index_df = pd.read_csv(index_path) if index_path.exists() else None
    if index_df is not None:
        index_df = index_df.iloc[:test_ds.x.shape[0]].reset_index(drop=True)
    plot_random_samples(
        test_ds.x,
        y_true_norm,
        y_pred_norm,
        mean,
        std,
        index_df,
        CONFIG.figure_dir / "training" / f"{safe_name}_random_samples.png",
        CONFIG.plot_samples,
        model_name,
    )

    metrics = {
        "model": model_name,
        "run_idx": run_idx,
        "target_col": CONFIG.target_col,
        "frequency": CONFIG.freq,
        "frequency_minutes": freq_minutes,
        "input_len": input_len,
        "pred_len": pred_len,
        "input_horizon_hours": input_len * freq_minutes / 60,
        "prediction_horizon_hours": pred_len * freq_minutes / 60,
        "n_params": n_params,
        "best_epoch": int(best_epoch),
        "best_val_mse_normalized": best_val,
        "test_mse_loss": float(test_loss),
        "test_metrics_normalized": norm_metrics,
        "test_metrics_original_n_flows": raw_metrics,
        "output_dir": str(out_dir),
    }
    (out_dir / "metrics.json").write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"{model_name} done | val_mse={best_val:.6f} | test_mse={norm_metrics['MSE']:.6f} | raw_mae={raw_metrics['MAE']:.3f}")
    return metrics, {
        "history": dict(history),
        "y_pred_norm": y_pred_norm,
        "y_true_norm": y_true_norm,
        "y_pred_raw": y_pred_raw,
        "y_true_raw": y_true_raw,
        "x_test_norm": test_ds.x,
        "mean": mean,
        "std": std,
        "index_df": index_df,
    }


def flatten_metrics(metrics):
    norm_m = metrics["test_metrics_normalized"]
    raw_m = metrics["test_metrics_original_n_flows"]
    return {
        "model": metrics["model"],
        "best_run_idx": metrics["run_idx"],
        "best_epoch": metrics["best_epoch"],
        "n_params": metrics["n_params"],
        "best_val_mse_normalized": metrics["best_val_mse_normalized"],
        "norm_MAE": norm_m["MAE"],
        "norm_MSE": norm_m["MSE"],
        "norm_RMSE": norm_m["RMSE"],
        "norm_MAPE(%)": norm_m["MAPE(%)"],
        "raw_MAE": raw_m["MAE"],
        "raw_MSE": raw_m["MSE"],
        "raw_RMSE": raw_m["RMSE"],
        "raw_MAPE(%)": raw_m["MAPE(%)"],
        "output_dir": metrics["output_dir"],
    }


def run_all_experiments():
    max_samples = CONFIG.smoke_max_samples if CONFIG.is_debug else None
    all_rows = []
    predictions = {}

    for model_name in CONFIG.run_models:
        run_records = []
        pred_records = []
        for run_idx in range(CONFIG.n_runs):
            metrics, pred_pack = run_one_model(model_name, run_idx, max_samples=max_samples)
            run_records.append(metrics)
            pred_records.append(pred_pack)
        best_i = int(np.argmin([m["best_val_mse_normalized"] for m in run_records]))
        best_metrics = run_records[best_i]
        predictions[model_name] = pred_records[best_i]
        row = flatten_metrics(best_metrics)
        row["experiment_group"] = (
            "comparison+ablation"
            if model_name in CONFIG.comparison_models and model_name in CONFIG.ablation_models
            else "comparison"
            if model_name in CONFIG.comparison_models
            else "ablation"
        )
        all_rows.append(row)

    results_df = pd.DataFrame(all_rows).sort_values("norm_MSE").reset_index(drop=True)
    results_df.to_csv(CONFIG.metric_dir / "experiment_summary.csv", index=False)
    (CONFIG.metric_dir / "experiment_summary.json").write_text(
        results_df.to_json(orient="records", force_ascii=False, indent=2), encoding="utf-8"
    )
    print(f"实验汇总已保存到: {CONFIG.metric_dir / 'experiment_summary.csv'}")
    return results_df, predictions


## 6. 可视化生成函数


In [7]:
MODEL_COLORS = {
    "DLinear": "#4C78A8",
    "Transformer": "#F2CF5B",
    "Autoformer": "#E45756",
    "TimeXer": "#72B7B2",
    "FEDformer": "#54A24B",
    "iTransformer": "#B279A2",
    "PatchTST": "#FF9DA6",
    "SDF-PatchTST": "#F58518",
    "SDF-PatchTST w/o Decomp": "#A0CBE8",
    "SDF-PatchTST w/o Spectral": "#BAB0AC",
}


def metric_barplot(df, models, metric_cols, out_path, title):
    sub = df[df["model"].isin(models)].copy()
    sub["model"] = pd.Categorical(sub["model"], categories=models, ordered=True)
    sub = sub.sort_values("norm_MSE", ascending=True)
    panel_width = max(6.0, 0.85 * max(len(sub), 1) + 2.0)
    fig, axes = plt.subplots(1, len(metric_cols), figsize=(panel_width * len(metric_cols), 5.5))
    if len(metric_cols) == 1:
        axes = [axes]
    for ax, metric_col in zip(axes, metric_cols):
        values = sub[metric_col].values
        labels = sub["model"].astype(str).values
        x = np.arange(len(sub))
        colors = [MODEL_COLORS.get(m, "#4C78A8") for m in labels]
        ax.bar(x, values, color=colors, alpha=0.88)
        if len(values) > 0:
            ax.set_ylim(0, max(values) * 1.16)
        for xi, value in enumerate(values):
            ax.text(xi, value, f"{value:.4f}", va="bottom", ha="center", fontsize=8, rotation=0)
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=35, ha="right")
        ax.set_ylabel(metric_col + " (lower is better)")
        ax.grid(axis="y", alpha=0.25)
    fig.suptitle(title)
    savefig(out_path)


def plot_metric_heatmap(df, models, out_path, title):
    sub = df[df["model"].isin(models)].set_index("model")
    metric_cols = ["norm_MAE", "norm_MSE", "norm_RMSE", "raw_MAE", "raw_RMSE"]
    mat = sub.loc[[m for m in models if m in sub.index], metric_cols]
    plt.figure(figsize=(9, max(4, 0.45 * len(mat) + 2)))
    if sns is not None:
        sns.heatmap(mat, annot=True, fmt=".4g", cmap="YlGnBu_r", linewidths=0.2)
    else:
        plt.imshow(mat.values, cmap="YlGnBu_r")
        plt.xticks(range(len(mat.columns)), mat.columns, rotation=45, ha="right")
        plt.yticks(range(len(mat.index)), mat.index)
        plt.colorbar()
    plt.title(title)
    savefig(out_path)


def plot_params_vs_mse(df, out_path):
    plt.figure(figsize=(8, 5))
    for _, row in df.iterrows():
        plt.scatter(row["n_params"], row["norm_MSE"], s=100, color=MODEL_COLORS.get(row["model"], "#4C78A8"))
        plt.text(row["n_params"], row["norm_MSE"], row["model"], fontsize=8, ha="left", va="bottom")
    plt.xscale("symlog")
    plt.xlabel("Trainable Parameters")
    plt.ylabel("Normalized MSE")
    plt.title("Model Size vs Test MSE")
    plt.grid(alpha=0.25)
    savefig(out_path)


def plot_validation_curves(predictions, models, out_path):
    plt.figure(figsize=(10, 5))
    for model_name in models:
        hist = predictions.get(model_name, {}).get("history", {})
        val_loss = hist.get("val_loss", [])
        if len(val_loss) == 0:
            continue
        plt.plot(val_loss, label=model_name, linewidth=1.8, color=MODEL_COLORS.get(model_name))
    plt.xlabel("Epoch")
    plt.ylabel("Validation MSE")
    plt.title("Validation Curves")
    plt.legend(ncol=2, fontsize=8)
    plt.grid(alpha=0.25)
    savefig(out_path)


def plot_true_pred_scatter(y_true, y_pred, out_path):
    true = y_true.reshape(-1)
    pred = y_pred.reshape(-1)
    if len(true) > CONFIG.max_scatter_points:
        rng = np.random.default_rng(CONFIG.seed)
        idx = rng.choice(len(true), CONFIG.max_scatter_points, replace=False)
        true = true[idx]
        pred = pred[idx]
    lo = min(true.min(), pred.min())
    hi = max(true.max(), pred.max())
    plt.figure(figsize=(6, 6))
    plt.scatter(true, pred, s=4, alpha=0.25, color="#4C78A8")
    plt.plot([lo, hi], [lo, hi], color="#E45756", linewidth=1.5)
    plt.xlabel("True n_flows")
    plt.ylabel("Predicted n_flows")
    plt.title("True vs Predicted")
    plt.grid(alpha=0.25)
    savefig(out_path)


def plot_residual_distribution(y_true, y_pred, out_path):
    residual = (y_pred - y_true).reshape(-1)
    plt.figure(figsize=(8, 4.5))
    plt.hist(residual, bins=100, color="#4C78A8", alpha=0.85)
    plt.axvline(0, color="#E45756", linewidth=1.5)
    plt.xlabel("Prediction Residual")
    plt.ylabel("Count")
    plt.title("Residual Distribution")
    plt.grid(alpha=0.25)
    savefig(out_path)


def plot_horizon_errors(y_true, y_pred, out_path):
    err = y_pred - y_true
    mae = np.mean(np.abs(err), axis=0)
    rmse = np.sqrt(np.mean(err ** 2, axis=0))
    bias = np.mean(err, axis=0)
    horizon = np.arange(1, y_true.shape[1] + 1) * CONFIG.freq_minutes / 60
    plt.figure(figsize=(10, 4.8))
    plt.plot(horizon, mae, label="MAE", linewidth=1.8)
    plt.plot(horizon, rmse, label="RMSE", linewidth=1.8)
    plt.plot(horizon, bias, label="Bias", linewidth=1.5)
    plt.axhline(0, color="black", linewidth=0.8)
    plt.xlabel("Forecast Horizon (hours)")
    plt.ylabel("Error")
    plt.title("Forecast Horizon Error Curve")
    plt.legend()
    plt.grid(alpha=0.25)
    savefig(out_path)


def plot_window_mae(y_true, y_pred, index_df, out_path):
    window_mae = np.mean(np.abs(y_pred - y_true), axis=1)
    x = index_df["target_start_step"].values if index_df is not None else np.arange(len(window_mae))
    plt.figure(figsize=(12, 4))
    plt.plot(x, window_mae, linewidth=1.0, color="#F58518")
    plt.xlabel("Target Start Step")
    plt.ylabel("Window MAE")
    plt.title("Test Window MAE Timeline")
    plt.grid(alpha=0.25)
    savefig(out_path)


def plot_abs_error_heatmap(y_true, y_pred, out_path, max_windows=160):
    abs_err = np.abs(y_pred - y_true)
    if abs_err.shape[0] > max_windows:
        abs_err = abs_err[:max_windows]
    plt.figure(figsize=(10, 6))
    if sns is not None:
        sns.heatmap(abs_err, cmap="magma", cbar_kws={"label": "Absolute Error"})
    else:
        plt.imshow(abs_err, aspect="auto", cmap="magma")
        plt.colorbar(label="Absolute Error")
    plt.xlabel("Forecast Step")
    plt.ylabel("Test Window")
    plt.title("Absolute Error Heatmap")
    savefig(out_path)


def plot_reconstructed_timeline(y_true, y_pred, index_df, out_path):
    if index_df is None:
        return
    step_to_true = {}
    step_to_preds = defaultdict(list)
    for i, row in index_df.iterrows():
        start = int(row["target_start_step"])
        for h in range(y_true.shape[1]):
            step = start + h
            step_to_true[step] = y_true[i, h]
            step_to_preds[step].append(y_pred[i, h])
    steps = np.array(sorted(step_to_true))
    true_line = np.array([step_to_true[s] for s in steps])
    pred_line = np.array([np.mean(step_to_preds[s]) for s in steps])
    plt.figure(figsize=(14, 4))
    plt.plot(steps, true_line, label="True", linewidth=1.3)
    plt.plot(steps, pred_line, label="Pred(avg overlap)", linewidth=1.3)
    plt.xlabel("Time Step")
    plt.ylabel(CONFIG.target_col)
    plt.title("Reconstructed Test Timeline")
    plt.legend()
    plt.grid(alpha=0.25)
    savefig(out_path)


def plot_high_error_samples(pack, out_path, n_samples=4):
    y_true = pack["y_true_raw"]
    y_pred = pack["y_pred_raw"]
    x_norm = pack["x_test_norm"]
    mean = pack["mean"]
    std = pack["std"]
    index_df = pack["index_df"]
    window_mae = np.mean(np.abs(y_pred - y_true), axis=1)
    top_ids = np.argsort(window_mae)[-n_samples:][::-1]
    fig, axes = plt.subplots(len(top_ids), 1, figsize=(10, 3.5 * len(top_ids)), squeeze=False)
    for ax, sid in zip(axes[:, 0], top_ids):
        hist = inverse_target(x_norm[sid, :, 0], mean, std)
        true = y_true[sid]
        pred = y_pred[sid]
        ax.plot(np.arange(-len(hist) + 1, 1), hist, label="History", linewidth=1.5)
        ax.plot(np.arange(1, len(true) + 1), true, label="True", linewidth=1.7)
        ax.plot(np.arange(1, len(pred) + 1), pred, label="Pred", linewidth=1.7)
        ax.axvline(0, color="gray", linestyle="--", linewidth=1)
        title = f"High Error Sample #{sid} | window_MAE={window_mae[sid]:.3f}"
        if index_df is not None and sid < len(index_df):
            title += f" | target_start_step={index_df.iloc[sid]['target_start_step']}"
        ax.set_title(title)
        ax.set_xlabel("10-minute step")
        ax.set_ylabel(CONFIG.target_col)
        ax.legend()
        ax.grid(alpha=0.25)
    savefig(out_path)


def plot_top_model_radar(df, out_path, top_k=5):
    sub = df.sort_values("norm_MSE").head(top_k).copy()
    metric_cols = ["norm_MAE", "norm_MSE", "norm_RMSE", "raw_MAE", "raw_RMSE"]
    values = sub[metric_cols].copy()
    # 指标越低越好，转为 0~1 的展示分数。
    score = 1 - (values - values.min()) / (values.max() - values.min() + 1e-12)
    labels = metric_cols
    angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
    angles += angles[:1]
    fig = plt.figure(figsize=(7, 7))
    ax = plt.subplot(111, polar=True)
    for i, row in score.iterrows():
        vals = row.tolist() + row.tolist()[:1]
        model_name = sub.loc[i, "model"]
        ax.plot(angles, vals, label=model_name, linewidth=1.8, color=MODEL_COLORS.get(model_name))
        ax.fill(angles, vals, alpha=0.08, color=MODEL_COLORS.get(model_name))
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels)
    ax.set_yticklabels([])
    ax.set_title("Top Model Radar Score")
    ax.legend(loc="upper right", bbox_to_anchor=(1.25, 1.1))
    savefig(out_path)


def create_result_visualizations(results_df, predictions):
    result_dir = CONFIG.figure_dir / "results"
    diag_dir = CONFIG.figure_dir / "diagnostics"

    metric_barplot(
        results_df,
        CONFIG.comparison_models,
        ["norm_MAE", "norm_MSE"],
        result_dir / "01_comparison_normalized_mae_mse.png",
        "Comparison Experiment: Normalized MAE / MSE",
    )
    metric_barplot(
        results_df,
        CONFIG.ablation_models,
        ["norm_MAE", "norm_MSE"],
        result_dir / "02_ablation_normalized_mae_mse.png",
        "Ablation Experiment: Normalized MAE / MSE",
    )
    metric_barplot(
        results_df,
        CONFIG.comparison_models,
        ["norm_MSE"],
        result_dir / "03_comparison_mse_ranking.png",
        "Comparison Experiment: Normalized MSE Ranking",
    )
    metric_barplot(
        results_df,
        CONFIG.ablation_models,
        ["norm_MSE"],
        result_dir / "04_ablation_mse_ranking.png",
        "Ablation Experiment: Normalized MSE Ranking",
    )
    plot_metric_heatmap(results_df, CONFIG.comparison_models, result_dir / "05_comparison_metric_heatmap.png", "Comparison Metric Heatmap")
    plot_metric_heatmap(results_df, CONFIG.ablation_models, result_dir / "06_ablation_metric_heatmap.png", "Ablation Metric Heatmap")
    plot_params_vs_mse(results_df, result_dir / "07_params_vs_mse.png")
    plot_validation_curves(predictions, CONFIG.run_models, result_dir / "08_validation_curves_overlay.png")
    plot_top_model_radar(results_df, result_dir / "09_top_model_radar.png")

    best_model = results_df.iloc[0]["model"]
    best_pack = predictions[best_model]
    plot_true_pred_scatter(best_pack["y_true_raw"], best_pack["y_pred_raw"], diag_dir / "01_true_vs_pred_scatter.png")
    plot_residual_distribution(best_pack["y_true_raw"], best_pack["y_pred_raw"], diag_dir / "02_residual_distribution.png")
    plot_horizon_errors(best_pack["y_true_raw"], best_pack["y_pred_raw"], diag_dir / "03_horizon_error_curve.png")
    plot_window_mae(best_pack["y_true_raw"], best_pack["y_pred_raw"], best_pack["index_df"], diag_dir / "04_window_mae_timeline.png")
    plot_abs_error_heatmap(best_pack["y_true_raw"], best_pack["y_pred_raw"], diag_dir / "05_absolute_error_heatmap.png")
    plot_reconstructed_timeline(best_pack["y_true_raw"], best_pack["y_pred_raw"], best_pack["index_df"], diag_dir / "06_reconstructed_test_timeline.png")
    plot_high_error_samples(best_pack, diag_dir / "07_high_error_samples.png")

    print(f"结果可视化已保存到: {result_dir}")
    print(f"最优模型诊断图已保存到: {diag_dir}")


## 7. 一键运行全部实验


In [8]:
RUN_FULL_EXPERIMENT = CONFIG.run_full_experiment

if RUN_FULL_EXPERIMENT:
    set_seed(CONFIG.seed)
    data_pack = prepare_forecast_data(force_rebuild=CONFIG.force_rebuild_data)
    print(json.dumps(data_pack["window_summary"]["splits"], ensure_ascii=False, indent=2))
    display(pd.DataFrame(data_pack["window_summary"]["splits"]).T)

    plot_data_visualizations(data_pack)

    results_df, predictions = run_all_experiments()
    display(results_df)

    create_result_visualizations(results_df, predictions)

    print("\n全部流程完成。")
    print("Summary CSV:", CONFIG.metric_dir / "experiment_summary.csv")
    print("Figures:", CONFIG.figure_dir)
    print("Runs:", CONFIG.run_dir)
else:
    print("RUN_FULL_EXPERIMENT=False，仅完成函数定义。若要一键运行全部流程，请设置 CONFIG.run_full_experiment=True 后从头运行。")


复用已存在的预处理数据和窗口样本。
{
  "train": {
    "rows": 28208,
    "samples": 27969,
    "x_shape": [
      27969,
      168,
      1
    ],
    "y_shape": [
      27969,
      72
    ],
    "first_window": {
      "input_start_step": 0,
      "input_end_step": 167,
      "target_start_step": 168,
      "target_end_step": 239,
      "input_start_time": "2000-01-01T00:00:00.000000000",
      "input_end_time": "2000-01-02T03:50:00.000000000",
      "target_start_time": "2000-01-02T04:00:00.000000000",
      "target_end_time": "2000-01-02T15:50:00.000000000"
    },
    "last_window": {
      "input_start_step": 27968,
      "input_end_step": 28135,
      "target_start_step": 28136,
      "target_end_step": 28207,
      "input_start_time": "2000-07-13T05:20:00.000000000",
      "input_end_time": "2000-07-14T09:10:00.000000000",
      "target_start_time": "2000-07-14T09:20:00.000000000",
      "target_end_time": "2000-07-14T21:10:00.000000000"
    }
  },
  "val": {
    "rows": 6044,
    "samples": 580

,rows,samples,x_shape,y_shape,first_window,last_window
train,28208,27969,"[27969, 168, 1]","[27969, 72]","{'input_start_step': 0, 'input_end_step': 167,...","{'input_start_step': 27968, 'input_end_step': ..."
val,6044,5805,"[5805, 168, 1]","[5805, 72]","{'input_start_step': 28208, 'input_end_step': ...","{'input_start_step': 34012, 'input_end_step': ..."
test,6046,5807,"[5807, 168, 1]","[5807, 72]","{'input_start_step': 34252, 'input_end_step': ...","{'input_start_step': 40058, 'input_end_step': ..."


数据可视化已保存到: /data2/hjs/pythonProject/pythonProject/Project4_wangluoliuliang/outputs/network_traffic_forecast/figures/data
\n================================================================================\nDLinear | run 00\n================================================================================
params=24,336 | train=27969 | val=5805 | test=5807
input=168 steps (28.0h), pred=72 steps (12.0h)


epoch=001 train_mse=0.85073767 val_mse=0.93631405 lr=1.000e-04


epoch=002 train_mse=0.64210862 val_mse=0.82936779 lr=1.000e-04


...

epoch=010 train_mse=0.42306928 val_mse=0.85374865 lr=5.000e-05


epoch=011 train_mse=0.41809949 val_mse=0.88182182 lr=2.500e-05
early_stop epoch=011 best_epoch=003 best_val_mse=0.69935234


SDF-PatchTST w/o Spectral done | val_mse=0.699352 | test_mse=0.379241 | raw_mae=765.345
实验汇总已保存到: /data2/hjs/pythonProject/pythonProject/Project4_wangluoliuliang/outputs/network_traffic_forecast/metrics/experiment_summary.csv


,model,best_run_idx,best_epoch,n_params,best_val_mse_normalized,norm_MAE,norm_MSE,norm_RMSE,norm_MAPE(%),raw_MAE,raw_MSE,raw_RMSE,raw_MAPE(%),output_dir,experiment_group
0,SDF-PatchTST w/o Decomp,0,4,454429,0.664625,0.427715,0.372456,0.610292,315.259433,769.567749,1419237.250,1191.317383,18.291238,/data2/hjs/pythonProject/pythonProject/Project...,ablation
1,PatchTST,0,4,454344,0.664523,0.427932,0.372600,0.610409,316.508222,769.855713,1418900.750,1191.176147,18.313287,/data2/hjs/pythonProject/pythonProject/Project...,comparison+ablation
2,SDF-PatchTST,0,3,456861,0.698026,0.421658,0.378937,0.615578,300.107598,764.743225,1465660.125,1210.644531,17.987654,/data2/hjs/pythonProject/pythonProject/Project...,comparison+ablation
3,SDF-PatchTST w/o Spectral,0,3,456776,0.699352,0.421920,0.379241,0.615825,300.175524,765.344910,1467648.875,1211.465576,17.993993,/data2/hjs/pythonProject/pythonProject/Project...,ablation
4,iTransformer,0,10,296136,0.619576,0.427947,0.385958,0.621255,269.855189,766.368469,1470544.000,1212.659912,17.730293,/data2/hjs/pythonProject/pythonProject/Project...,comparison
5,TimeXer,0,4,463688,0.711881,0.450361,0.408933,0.639479,314.063168,799.414124,1527267.000,1235.826416,18.751943,/data2/hjs/pythonProject/pythonProject/Project...,comparison
6,Autoformer,0,16,1781456,0.516955,0.490457,0.437917,0.661753,437.579918,847.751892,1543371.000,1242.324829,20.241778,/data2/hjs/pythonProject/pythonProject/Project...,comparison
7,Transformer,0,3,1835336,0.632031,0.472119,0.440635,0.663803,341.975451,836.432373,1635357.625,1278.811035,19.232358,/data2/hjs/pythonProject/pythonProject/Project...,comparison
8,DLinear,0,19,24336,0.715537,0.466396,0.455952,0.675242,252.609825,861.928650,1859021.625,1363.459473,19.012891,/data2/hjs/pythonProject/pythonProject/Project...,comparison
9,FEDformer,0,13,24400,0.725062,0.467648,0.459567,0.677914,248.253083,865.346863,1878565.000,1370.607544,19.053081,/data2/hjs/pythonProject/pythonProject/Project...,comparison


结果可视化已保存到: /data2/hjs/pythonProject/pythonProject/Project4_wangluoliuliang/outputs/network_traffic_forecast/figures/results
最优模型诊断图已保存到: /data2/hjs/pythonProject/pythonProject/Project4_wangluoliuliang/outputs/network_traffic_forecast/figures/diagnostics

全部流程完成。
Summary CSV: /data2/hjs/pythonProject/pythonProject/Project4_wangluoliuliang/outputs/network_traffic_forecast/metrics/experiment_summary.csv
Figures: /data2/hjs/pythonProject/pythonProject/Project4_wangluoliuliang/outputs/network_traffic_forecast/figures
Runs: /data2/hjs/pythonProject/pythonProject/Project4_wangluoliuliang/outputs/network_traffic_forecast/runs
